# ⚠️ Historical timing baseline — AutoCarver **7.0.5**, single process

**Period artifact, not current API.** Exists only to measure the 7.0.5 carving
wall-clock against the 2026 re-run; it needs its own environment
(`uv venv --python 3.11 .venv-705 && uv pip install --python .venv-705 -r
requirements-705.txt` — a full freeze, because installing `autocarver==7.0.5`
today resolves a different scikit-learn and numpy than the measured run).


# Severity model — 2025 baseline, timed  *(AutoCarver 7.0.5)*

Verbatim copy of `amount_model.ipynb` cells 0–11 (load → stratified split on the
discretized amount → `Processor` → column typing → `ContinuousCarver`), with
**nothing changed but one `time.perf_counter()` wrapper**. Everything after the
carving step is dropped: this notebook exists only to measure the 7.0.5 carving
wall-clock against the 2026 run, on the same machine.

Note the 2025 notebook carves *only* qualitatives — its quantitative carving cell
was already commented out — so there is a single timing here.

# Loading Data & Processing

First, let's load the challenge's data and target and join them on ``ID``

In [1]:
import pandas as pd

data_path = "../data/"

# loading x_train
data = pd.read_csv(data_path + "train_input_Z61KlZo.csv")
data.set_index("ID", inplace=True)
print("x_train", data.shape)

# loading target
target = pd.read_csv(data_path + "train_output_DzPxaPY.csv")
target.set_index("ID", inplace=True)
print("y_train", target.shape)

# joining x_train and y_train
data = data.join(target.drop("ANNEE_ASSURANCE", axis=1))
print("data", data.shape)

<tmp>/ipykernel_15772\1250711303.py:6: DtypeWarning: Columns (16,17,29,30,31,126,128,129,132,133,135,138,371) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(data_path+'train_input_Z61KlZo.csv')


x_train (383610, 373)
y_train (383610, 4)


data (383610, 376)


## Stratified Sampling

**Stratified Sampling** is applied to ensure same distribution between train (80%) and dev (20%) samples

Stratification is done using classes that maximize association with the number of claims per observation. This allows samples to be similarly distributed on claim frequencies.

In [2]:
from AutoCarver import BinaryCarver
from sklearn.model_selection import train_test_split

target_col = "CM"

# loading target carver
target_carver = BinaryCarver.load("model/cm_carver_freq_tschuprowt.json")

# discretizing the target
y_freq = target_carver.transform(data)[target_col]

# Train-test split
x_train, x_dev, y_train, y_dev = train_test_split(
    data, data[target_col], test_size=0.2, random_state=42, stratify=y_freq
)
print("y_train mean", y_train.mean())
print("y_dev mean", y_dev.mean())

y_train mean 182.78860026567781
y_dev mean 181.45375929980975


In [3]:
print("amount mean per class")
(
    y_train.groupby(target_carver.transform(x_train)[target_col]).mean().sort_index(),
    y_dev.groupby(target_carver.transform(x_dev)[target_col]).mean().sort_index(),
)

amount mean per class


(CM
 0.0         0.634508
 1.0       726.987101
 2.0      1158.781582
 3.0     10554.699356
 4.0    197900.752676
 Name: CM, dtype: float64,
 CM
 0.0         0.570218
 1.0       731.756333
 2.0      1155.480222
 3.0     10198.466203
 4.0    198281.136167
 Name: CM, dtype: float64)

In [4]:
print("amount median per class")
(
    y_train.groupby(target_carver.transform(x_train)[target_col]).median().sort_index(),
    y_dev.groupby(target_carver.transform(x_dev)[target_col]).median().sort_index(),
)

amount median per class


(CM
 0.0         0.000
 1.0       723.900
 2.0      1131.300
 3.0      5313.735
 4.0    156541.820
 Name: CM, dtype: float64,
 CM
 0.0         0.000
 1.0       714.250
 2.0      1149.000
 3.0      5892.000
 4.0    179814.445
 Name: CM, dtype: float64)

# Feature Engineering

The `Processor` class is used to build several new features:
* `ZONE` is converted to `ZONE_REGION`
* Copies of `ALTITUDE_xxx`, `IND_xxx`, `MEN_xxx` and `LOG_xxx` are converted to numerical features: `ALTITUDE_xxx_num`, `IND_xxx_num`, `MEN_xxx_num` and `LOG_xxx_num`
* Vetusty of buildings is computed: `LOG_VETUSTE`
* It learns the distribution of `LOG_TOT`, `LOG_VETUSTE`, `MEN_TOT`, `IND_TOT`, `IND_SNV`,`ALTITUDE_TOT` per `ZONE_REGION`on train sample. It's then used to compute the ratio of each sample to its mean per region (sort of a measure of divergence from the regional mean)
* Per observation, `CA_TOT` and `CA_MEAN` are computed as the sum and mean of `CA1`, `CA2` and `CA3`. `CA_TOT` and `CA_MEAN` are used to compute ratios with  `CA1`, `CA2` and `CA3`
* Open Source data from [Base de Données sur les Incendies de Forêts en France](https://bdiff.agriculture.gouv.fr/) are added: fire extinction rates per `ZONE` in 2023, total surfaces burnt in 2023 and in the 2016-2020 period per `ZONE`, surface burnt over surface of forest per `ZONE` in 2023. Those features are crossed with `NB_CASERNES` and `ZONE_VENT`
* Numerical `KAPITAL_xxx` columns are sumed and maxed out into `KAPITAL_SUM` and `KAPITAL_MAX`
* Temperature columns are crossed with one another
* Binary non-numerical columns are one hot encoded

In [5]:
import warnings
from data_toolkit import Processor

warnings.simplefilter(action="ignore", category=FutureWarning)

proc = Processor()
x_train = proc.fit_transform(x_train)
x_dev = proc.transform(x_dev)
data = proc.transform(data)
data.shape, x_train.shape, x_dev.shape

((383610, 562), (306888, 562), (76722, 562))

# Feature Processing

## Sorting features per data type

* numerical features
* categorical features
* ordinal features (with their respective ordering)

In [6]:

# get the columns that are categorical
categorical_columns = x_train.select_dtypes(include=["object"]).columns

# get the columns that are numerical
numerical_columns = x_train.select_dtypes(include=["int64", "float64"]).columns

# getting ordinal columns
ordinals = [
    "NB_CASERNES",
    "BDTOPO_BAT_MAX_HAUTEUR",
    "HAUTEUR_MAX",
    "HAUTEUR",
    "BDTOPO_BAT_MAX_HAUTEUR_MAX",
    "MEN_SURF",
    "IND_SNV",
    "IND_INC",
    "IND_Y9",
    "IND_0_Y1",
    "IND",
    "LOG_SOC",
    "LOG_INC",
    "LOG_APA3",
    "LOG_AVA1",
    "MEN_MAIS",
    "MEN_COLL",
    "MEN_FMP",
    "MEN_PROP",
    "MEN_PAUV",
    "MEN",
    "COEFASS",
]
ordinals += [
    "DISTANCE_111",
    "DISTANCE_112",
    "DISTANCE_121",
    "DISTANCE_122",
    "DISTANCE_123",
    "DISTANCE_124",
    "DISTANCE_131",
    "DISTANCE_132",
    "DISTANCE_133",
    "DISTANCE_141",
    "DISTANCE_142",
    "DISTANCE_211",
    "DISTANCE_212",
    "DISTANCE_213",
    "DISTANCE_221",
    "DISTANCE_222",
    "DISTANCE_223",
    "DISTANCE_231",
    "DISTANCE_242",
    "DISTANCE_243",
    "DISTANCE_244",
    "DISTANCE_311",
    "DISTANCE_312",
    "DISTANCE_313",
    "DISTANCE_321",
    "DISTANCE_322",
    "DISTANCE_323",
    "DISTANCE_324",
    "DISTANCE_331",
    "DISTANCE_332",
    "DISTANCE_333",
    "DISTANCE_334",
    "DISTANCE_335",
    "DISTANCE_411",
    "DISTANCE_412",
    "DISTANCE_421",
    "DISTANCE_422",
    "DISTANCE_423",
    "DISTANCE_511",
    "DISTANCE_512",
    "DISTANCE_521",
    "DISTANCE_522",
    "DISTANCE_523",
    "PROPORTION_11",
    "PROPORTION_12",
    "PROPORTION_13",
    "PROPORTION_14",
    "PROPORTION_21",
    "PROPORTION_22",
    "PROPORTION_23",
    "PROPORTION_24",
    "PROPORTION_31",
    "PROPORTION_32",
    "PROPORTION_33",
    "PROPORTION_41",
    "PROPORTION_42",
    "PROPORTION_51",
    "PROPORTION_52",
    "MEN_1IND",
    "MEN_5IND",
    "LOG_A1_A2",
    "LOG_A2_A3",
    "IND_Y1_Y2",
    "IND_Y2_Y3",
    "IND_Y3_Y4",
    "IND_Y4_Y5",
    "IND_Y5_Y6",
    "IND_Y6_Y7",
    "IND_Y7_Y8",
    "IND_Y8_Y9",
    "DISTANCE_1",
    "DISTANCE_2",
    "ALTITUDE_1",
    "ALTITUDE_2",
    "ALTITUDE_3",
    "ALTITUDE_4",
    "ALTITUDE_5",
    "NBJTX25_MM_A",
    "NBJTX25_MMAX_A",
    "NBJTX25_MSOM_A",
    "NBJTX0_MM_A",
    "NBJTX0_MMAX_A",
    "NBJTX0_MSOM_A",
    "NBJTXI27_MM_A",
    "NBJTXI27_MMAX_A",
    "NBJTXI27_MSOM_A",
    "NBJTXS32_MM_A",
    "NBJTXS32_MMAX_A",
    "NBJTXS32_MSOM_A",
    "NBJTXI20_MM_A",
    "NBJTXI20_MMAX_A",
    "NBJTXI20_MSOM_A",
    "NBJTX30_MM_A",
    "NBJTX30_MMAX_A",
    "NBJTX30_MSOM_A",
    "NBJTX35_MM_A",
    "NBJTX35_MMAX_A",
    "NBJTX35_MSOM_A",
    "NBJTN10_MM_A",
    "NBJTN10_MMAX_A",
    "NBJTN10_MSOM_A",
    "NBJTNI10_MM_A",
    "NBJTNI10_MMAX_A",
    "NBJTNI10_MSOM_A",
    "NBJTN5_MM_A",
    "NBJTN5_MMAX_A",
    "NBJTN5_MSOM_A",
    "NBJTNS25_MM_A",
    "NBJTNS25_MMAX_A",
    "NBJTNS25_MSOM_A",
    "NBJTNI15_MM_A",
    "NBJTNI15_MMAX_A",
    "NBJTNI15_MSOM_A",
    "NBJTNI20_MM_A",
    "NBJTNI20_MMAX_A",
    "NBJTNI20_MSOM_A",
    "NBJTNS20_MM_A",
    "NBJTNS20_MMAX_A",
    "NBJTNS20_MSOM_A",
    "NBJTMS24_MM_A",
    "NBJTMS24_MMAX_A",
    "NBJTMS24_MSOM_A",
    "TAMPLIAB_VOR_MM_A",
    "TAMPLIAB_VOR_MMAX_A",
    "TAMPLIM_VOR_MM_A",
    "TAMPLIM_VOR_MMAX_A",
    "TM_VOR_MM_A",
    "TM_VOR_MMAX_A",
    "TMM_VOR_MM_A",
    "TMM_VOR_MMAX_A",
    "TMMAX_VOR_MM_A",
    "TMMAX_VOR_MMAX_A",
    "TMMIN_VOR_MM_A",
    "TMMIN_VOR_MMAX_A",
    "TN_VOR_MM_A",
    "TN_VOR_MMAX_A",
    "TNAB_VOR_MM_A",
    "TNAB_VOR_MMAX_A",
    "TNMAX_VOR_MM_A",
    "TNMAX_VOR_MMAX_A",
    "TX_VOR_MM_A",
    "TX_VOR_MMAX_A",
    "TXAB_VOR_MM_A",
    "TXAB_VOR_MMAX_A",
    "TXMIN_VOR_MM_A",
    "TXMIN_VOR_MMAX_A",
    "NBJFF10_MM_A",
    "NBJFF10_MMAX_A",
    "NBJFF10_MSOM_A",
    "NBJFF16_MM_A",
    "NBJFF16_MMAX_A",
    "NBJFF16_MSOM_A",
    "NBJFF28_MM_A",
    "NBJFF28_MMAX_A",
    "NBJFF28_MSOM_A",
    "NBJFXI3S10_MM_A",
    "NBJFXI3S10_MMAX_A",
    "NBJFXI3S10_MSOM_A",
    "NBJFXI3S16_MM_A",
    "NBJFXI3S16_MMAX_A",
    "NBJFXI3S16_MSOM_A",
    "NBJFXI3S28_MM_A",
    "NBJFXI3S28_MMAX_A",
    "NBJFXI3S28_MSOM_A",
    "NBJFXY8_MM_A",
    "NBJFXY8_MMAX_A",
    "NBJFXY8_MSOM_A",
    "NBJFXY10_MM_A",
    "NBJFXY10_MMAX_A",
    "NBJFXY10_MSOM_A",
    "NBJFXY15_MM_A",
    "NBJFXY15_MMAX_A",
    "NBJFXY15_MSOM_A",
    "FFM_VOR_MM_A",
    "FFM_VOR_MMAX_A",
    "FXI3SAB_VOR_MM_A",
    "FXI3SAB_VOR_MMAX_A",
    "FXIAB_VOR_MM_A",
    "FXIAB_VOR_MMAX_A",
    "FXYAB_VOR_MM_A",
    "FXYAB_VOR_MMAX_A",
    "FFM_VOR_COM_MM_A_Y",
    "FFM_VOR_COM_MMAX_A_Y",
    "FXI3SAB_VOR_COM_MM_A_Y",
    "FXI3SAB_VOR_COM_MMAX_A_Y",
    "NBJRR50_MM_A",
    "NBJRR50_MMAX_A",
    "NBJRR50_MSOM_A",
    "NBJRR1_MM_A",
    "NBJRR1_MMAX_A",
    "NBJRR1_MSOM_A",
    "NBJRR5_MM_A",
    "NBJRR5_MMAX_A",
    "NBJRR5_MSOM_A",
    "NBJRR10_MM_A",
    "NBJRR10_MMAX_A",
    "NBJRR10_MSOM_A",
    "NBJRR30_MM_A",
    "NBJRR30_MMAX_A",
    "NBJRR30_MSOM_A",
    "NBJRR100_MM_A",
    "NBJRR100_MMAX_A",
    "NBJRR100_MSOM_A",
    "RR_VOR_MM_A",
    "RR_VOR_MMAX_A",
    "RRAB_VOR_MM_A",
    "RRAB_VOR_MMAX_A",
]
# ordinals += ["AN_EXERC"]
ordinals += ["TAILLE1", "TAILLE2"]
ordinal_columns = {
    col: list(data[col].value_counts().sort_index().index)
    for col in ordinals
    if col in data.columns
}
ordinal_columns["PROPORTION_32"] += ["10. > 90"]
ordinal_columns.update(
    {
        "CARACT4": [
            "absence de surface",
            "Surface de moins d",
            "Surface entre 501",
            "Surface entre 1001",
            "Surface entre 1501",
            "Surface de plus de",
        ],
        "SURFACE4": [
            "0",
            "500",
            "1000",
            "1500",
            "2000",
            "2500",
            "3000",
            "3500",
            "4000",
            "4500",
            "5000",
            "5500",
            "6000",
            "6500",
            "7000",
            "7000+",
        ],
        "SURFACE6": [
            "0",
            "500",
            "1000",
            "1500",
            "2000",
            "2500",
            "3000",
            "3500",
            "4000",
            "4500",
            "5000",
            "5500",
            "6000",
            "6500",
            "7000",
            "7000+",
        ],
        "total_surface_2023": [
            "Aucun feu",
            "<10ha",
            "10-20ha",
            "20-50ha",
            "50-100ha",
            "100-200ha",
            ">200ha",
        ],
        "total_surface_5y": [
            "Aucun feu",
            "<10ha",
            "10-20ha",
            "20-50ha",
            "50-100ha",
            "100-200ha",
            ">200ha",
        ],
        "surface_over_forest": [
            "Absence de feu",
            "<0.05",
            "0.05-0.1",
            "0.1-0.2",
            "0.5-2",
        ],
        "fire_extinction_rates": ["Aucun feu", "<50%", "50-70%", "70-85%", ">85%"],
    }
)

# get the columns that are to be removed
to_remove = target.columns.tolist() + [target_col]
to_remove += [c for c in data.columns if "MMSOM" in c]
to_remove += [
    "DEROG3",
    "DEROG13",
    "DEROG16",
    "DEROG8",
    "DEROG14",
    "TARGET",
]  # no values
to_remove += [
    "DEROG13_formatted",
    "DEROG8_formatted",
    "DEROG3_formatted",
    "DEROG16_formatted",
    "DEROG14_formatted",
]
to_remove += ["IND_Y1_Y2_num", "IND_INC_num"]

# removing columns
categorical_columns = [
    col
    for col in categorical_columns
    if col not in to_remove and col not in ordinal_columns
]
categorical_columns += ["TYPERS"]
numerical_columns = [
    col
    for col in numerical_columns
    if col not in to_remove
    and col not in ordinal_columns
    and col not in categorical_columns
]
print(
    len(categorical_columns),
    len(numerical_columns),
    len(ordinal_columns),
    len(categorical_columns) + len(numerical_columns) + len(ordinal_columns),
)

193 115 238 546


## Processing Qualitative Features

For a continuous target variable, following processing is applied:
* Pre-processing of ordinals:
    - ordering modalities according to user-provided values
    - grouping modalities with less than `min_freq=3%` frequency into their closest modality (previous or next modality) according to target mean (train sample)
* Pre-processing of categoricals:
    - grouping modalities with less than `min_freq=3%` frequency into a dedicated one (train sample)
    - ordering modalities according to target mean (train sample)
* All combinations of up to `max_n_mod=5` modalities are sorted by Kruskall-'s T with the target variable (train sample)
* Robustness of each combination is put to test (dev sample) 
    - representativness of modalities (more than `min_freq/3=1.5%` frequency)
    - distinct target mean per consecutive modalities
    - no inversion of target means between train and dev modalities

For multiclass target variables, the binary-oriented processing steps are applied to each class of the target variable with a One vs Rest approach (except for one of the classes)

In [7]:
import time

_t0 = time.perf_counter()

In [8]:
from AutoCarver import ContinuousCarver, Features

# defining the features to carve
features = Features(categoricals=categorical_columns, ordinals=ordinal_columns)

# defining the carver
carver = ContinuousCarver(
    features=features,
    min_freq=0.03,
    max_n_mod=5,
    dropna=False,
    copy=False,
    verbose=False,
)

# carving train data and testing robustness on dev data
x_train = carver.fit_transform(x_train, y_train, X_dev=x_dev, y_dev=y_dev)

In [9]:
print(
    "[2025] qualitative carving (single-process): %.1fs" % (time.perf_counter() - _t0)
)

[2025] qualitative carving (single-process): 4724.7s
